In [ ]:
!apt-get -qq install aria2 > /dev/null 2>&1
!pip install --quiet omnicloudmask==1.7.0

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cloudband.acquisition import zenodo
from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.eval.report import as_percentages, to_frame
from cloudband.labels import pixbox_l8
from cloudband.pipelines import pixbox_l8 as pipeline

reload_package("cloudband")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/data/pixbox_l8")
SCENES_DIR = Path("/content/data/scenes_l8")
PRED_DIR = Path("/content/data/predictions_l8")
RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)

In [ ]:
zenodo.download(zenodo.L8_LABELS, DATA_DIR)
!unzip -o -q {DATA_DIR}/{zenodo.L8_LABELS_ARCHIVE} -d {DATA_DIR}

table = pipeline.load_reference(DATA_DIR / zenodo.L8_LABELS_CSV)
print("pixels:", len(table))
print("counts:", pixbox_l8.label_counts(pixbox_l8.label_masks(table)))

In [ ]:
archive = zenodo.download(zenodo.L8_SCENES, Path("/content/data"))
!unzip -o -q {archive} -d {SCENES_DIR}
!ls {SCENES_DIR} | head
print("entradas:", len(list(SCENES_DIR.iterdir())))

In [ ]:
expected = sorted(pixbox_l8.PRODUCT_ID_TO_SCENE.values())
found = {p.name for p in SCENES_DIR.rglob("*") if p.is_dir()}
missing = [name for name in expected if not any(name in f for f in found)]
print("esperadas:", len(expected), "| faltando:", missing)
assert not missing, missing

In [ ]:
scene = sorted(SCENES_DIR.rglob(f"{expected[0]}*"))[0]
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)

masks = ocm.predict_scenes([scene], PRED_DIR, config, sensor=ocm.LANDSAT8)
pred = pipeline.read_prediction(masks[0])

pid = next(k for k, v in pixbox_l8.PRODUCT_ID_TO_SCENE.items() if v == expected[0])
sub = table[table[pixbox_l8.PRODUCT_COLUMN] == pid]
ref = pixbox_l8.label_masks(sub)
got = pixbox_l8.sample_predictions(sub, pred)

print("shape:", pred.shape)
print("referência:", {k: int(v.sum()) for k, v in ref.items()})
print("predição:  ", {c: int((got == c).sum()) for c in range(4)})

In [ ]:
scene_paths = [sorted(SCENES_DIR.rglob(f"{name}*"))[0] for name in expected]
masks = ocm.predict_scenes(scene_paths, PRED_DIR, config, sensor=ocm.LANDSAT8)
print("masks:", len(masks))

In [ ]:
scored = pipeline.attach_predictions(table, PRED_DIR)
confusions = pipeline.score(scored)
as_percentages(to_frame(confusions))